# Introduction to Optimization in Gurobi (Solutions)
The purpose of this exercise is to introduce you to mathematical optimization problems using the python API gurobipy.
- Section 1 presents how to install gurobi and necessary licenses and dependencies
- Section 2 introduces a simple example and exercise to familiarize yourselves with gurobi basics. 
- Section 3 introduces a more general way to formulate and solve optimization problems. 
- Section 4 gives a (very) basic introduction to object-oriented programming, and exemplifies how object-oriented programming can be used to structure optimization problems. 
- Section 5 is optional and focuses on sensitivity analysis and understanding the results

## Section 1. Installation Guide

### What is Gurobi/Gurobipy?

- Gurobi is a mathematical solver, combining advanced algorithms to solve numerically complex mathwematical optimization problems.
- Gurobipy is a python API used to formulate and solve mathematical optimization problems in python (similar to pyomo or JuMP in Julia).
- It is developed by the same company which develops the gurobi solver. 


### Installing Gurobipy


You can install gurobipy by running:

In [ ]:
#!pip install gurobipy
# Comment this line after installing gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 8.9 MB/s  0:00:01 eta 0:00:01

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Everytime you use Gurobi, you will need to import the package ```gurobipy```. The specific module ```GRB``` is commonly imported separately, as it is used frequently. 

In [7]:
# Import packages
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np 

### Free Academic LIcense

You then need to obtain a free Academic Named-User License, following these steps:

 1) Register for a free [Gurobi account as an academic and log in](https://portal.gurobi.com/iam/register/?_gl=1*ah7zi4*_up*MQ..*_gs*MQ..&gclid=CjwKCAjwlaTGBhANEiwAoRgXBb0o3PUl8z1tzZOZ9p3KbQPezzjJDyr4wWWdA-fs1K6uV5dppoNYihoCd98QAvD_BwE)
2) Visit the [Download Gurobi Optimizer page](https://www.gurobi.com/downloads/gurobi-software/?_gl=1*ah7zi4*_up*MQ..*_gs*MQ..&gclid=CjwKCAjwlaTGBhANEiwAoRgXBb0o3PUl8z1tzZOZ9p3KbQPezzjJDyr4wWWdA-fs1K6uV5dppoNYihoCd98QAvD_BwE) and download the version you need, as well as the README.txt.
3) Follow the instructions in README.txt to install the software.
4) Once installed, visit the [Gurobi User Portal]() to request your free **Named-User** License.
5) Next, run grbgetkey using the argument provided on the Academic License Detail page (ex: grbgetkey ae36ac20-16e6-acd2-f242-4da6e765fa0a). The grbgetkey program will prompt you to store the license file on your machine.

 **Note that you must be connected to DTU network or eduroam when downloading the academic license for the first time.**

If you encounter an “ERROR 303” message when running grbgetkey, please see the article, [How do I resolve an “ERROR 303” from grbgetkey?](https://support.gurobi.com/hc/en-us/articles/360038994471-How-do-I-resolve-an-ERROR-303-when-running-grbgetkey?_gl=1%2A2dyq1p%2A_up%2AMQ..%2A_gs%2AMQ..&gclid=CjwKCAjwlaTGBhANEiwAoRgXBb0o3PUl8z1tzZOZ9p3KbQPezzjJDyr4wWWdA-fs1K6uV5dppoNYihoCd98QAvD_BwE).

Your license will be valid for up to one year. You can request additional Academic Named-User licenses via the User Portal as long as you maintain eligibility.

This step-by-step video provides a detailed overview of the installation process: 

[![Watch on YouTube](https://img.youtube.com/vi/fRKhao2bzsY/hqdefault.jpg)](https://www.youtube.com/watch?v=fRKhao2bzsY)



## Section 2. Getting Started with Optimization in Gurobipy

### Simple example 

Let's use the following problem as an example:

$$
  \begin{align}
      \textrm{minimize} \quad &30x_1 + 20x_2 \\
      \textrm{subject to} \quad &0.6x_1 + 0.2x_2 \geq 60 \\
      &0.4x_1 + 0.8x_2 \geq 100 \\
      &x_1 \geq 0, x_2 \geq 0 \\
  \end{align}
$$

#### Initialize model

We initialize a model object in which we'll store the problem.

In [8]:
#create and name a new gurobi model
model = gp.Model(name="Toy Problem")

Set parameter Username
Set parameter LicenseID to value 2861715
Academic license - for non-commercial use only - expires 2027-09-09


#### Add elements to the model: variables, constraints, and objective 

- We can add variables to the model with the method ```model.addVar(lb=0.0, ub=float('inf'), vtype=GRB.CONTINUOUS, name="")```.
- You should specify the lower and upper bounds as well as domain using the arguments ```lb```, ```ub```, and ```vtype```, respectively.

<b>Note that the default lower bound is 0, and the default variable domain is continuous!<b>

In [4]:
# Note that these two variables have the same bounds and domain
x_1 = model.addVar(lb=0, ub=float('inf'), vtype=GRB.CONTINUOUS, name="x_1")
x_2 = model.addVar(name="x_2")

- Generally, we add constraints with the ```model.addConstr(lhs,direction,rhs,name="")``` method
- You can specify the expression of the left-hand side (```lhs```), ```direction``` ($>=$, $=$ or $<=$), and right-hand side (```rhs```) of the constraint as separate arguments. In the ```GRB```module, you can find the three signs ```GRB.GREATER_EQUAL```, ```GRB.EQUAL```, and ```GRB.LESS_EQUAL```.
- In this case, the constraints are linear so we can use the ```model.addLConstr(constr, name="")``` method.

In [5]:
constraint_1 = model.addLConstr(0.6*x_1 + 0.2*x_2, GRB.GREATER_EQUAL, 60, name='constraint_1')
constraint_2 = model.addLConstr(0.4*x_1 + 0.8*x_2, GRB.GREATER_EQUAL, 100, name='constraint_2')

<b>Note that it is important to store the variables and constraints in a meaningful way so you can easily access the values of the primal and dual variables after solving.<b>

- We define the objective function with the method ```model.setObjective(expr, sense=None)```. 
- You should specify the expression of the objective function **and the sense** of the optimization model, i.e. minimizing or maximizing. In the ```GRB``` module, you can find the two sense arguments ```GRB.MINIMIZE``` and ```GRB.MAXIMIZE```. 

In [6]:
model.setObjective(30*x_1 + 20*x_2, GRB.MINIMIZE)

- Now, we can solve the optimization problem with the method ```model.optimize```.

In [7]:
model.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.4 LTS")


CPU model: Intel(R) Xeon(R) Processor @ 2.10GHz, instruction set [SSE2|AVX|AVX2|AVX512]


Thread count: 1 physical cores, 1 logical processors, using up to 1 threads


Optimize a model with 2 rows, 2 columns and 4 nonzeros (Min)


Model fingerprint: 0x20d42a0c


...

Iteration    Objective       Primal Inf.    Dual Inf.      Time


       0    0.0000000e+00   1.600000e+02   0.000000e+00      0s


       2    3.9000000e+03   0.000000e+00   0.000000e+00      0s


Solved in 2 iterations and 0.01 seconds (0.00 work units)


Optimal objective  3.900000000e+03


- We can check whether the problem was solved to optimality with ```model.status```.
- If so, we retrieve optimal objective function with ```model.ObjVal``` 
- and optimal variable and Pi values with ```var.x``` and ```constr.Pi```, respectively. **We will discuss the meaning of the Pi values later**

In [8]:
model.status
model.ObjVal
x_1.x
x_2.x
constraint_1.Pi
constraint_2.Pi

15.0

#### Functions for checking status and printing results of LP

- To streamline the rest of the tutorial, we create a function ```LP_saver``` which checks the status and saves the results of any optimization model, and a function ```LP_printer``` which prints the results (objective value, decision variables, and **Pi** values).

**We will discuss later what these Pi values represent**

In [9]:
# create two functions that save and print the results of a standardized linear optimization problem solved with Gurobi

def LP_saver(
    model
):
    if model.status == GRB.OPTIMAL:
        optimal_variables = {v.VarName: v.x for v in model.getVars()} # Save optimal values of decision variables
        optimal_Pis = {c.ConstrName: c.Pi for c in model.getConstrs()} # Save optimal values of Pi values 
        optimal_objective= model.objVal # Save optimal value of objective function
    else: 
        optimal_variables = None
        optimal_Pis = None
        optimal_objective = None
        print("Optimization was not successful")
        
    return optimal_objective, optimal_variables, optimal_Pis

def LP_printer(
    model
):
    if model.status == GRB.OPTIMAL:
        print()
        print('-------------------   RESULTS  -------------------')
        print("Optimal objective value:", model.objVal)
        for v in model.getVars():
                print(f'Optimal value of {v.VarName}:', v.x)
        for c in model.getConstrs():
                print(f'Pi value associated with {c.ConstrName}:', c.Pi)
    else:
            print("Optimization was not successful")

- We can now check the status and print the solutions to any optimization problem:

In [10]:
LP_printer(model) #print the results of the optimization


-------------------   RESULTS  -------------------
Optimal objective value: 3900.0
Optimal value of x_1: 70.0
Optimal value of x_2: 90.0
Pi value associated with constraint_1: 40.0
Pi value associated with constraint_2: 15.0


These steps are summarized (using another example) in this short video: 

[![Watch on YouTube](https://img.youtube.com/vi/7sMhvLn02P8/hqdefault.jpg)](https://www.youtube.com/watch?v=7sMhvLn02P8&list=PLaxOs-8sLebsGEsuo1FEmpyM1LFrzmQBN&index=2)

### Task: Overnight Bike-Lane Clearing

Now, let's solve a small resource-allocation problem.

The municipality must clean snow from its bike lanes. The lanes are spread over several districts, and the total length must be cleaned before 6am. 
The municipality does not want to pay for cleaning more than what is needed.
It owns no machines, so the work is bought from a number of contractors. 
Each contractor charges a fixed price per kilometre and clears at its own speed, so none of them can clean more than a certain length – depending on its speed and the remaining hours before the deadline.

We define the following sets, parameters and decision variables:
- $N$: number of available contractors
- $D$: number of districts to be cleared
- $H$: hours available before the 6 am deadline (in h)
- $x_i$: kilometres cleared by contractor $i=1,...,N$ (in km)
- $c_i$: price of contractor $i=1,...,N$ (in DKK/km)
- $v_i$: clearing speed of contractor $i=1,...,N$ (in km/h)
- $L_j$: length of bike lane in district $j=1,...,D$ (in km)

**Note that the parameter $H$ converts a clearing speed (in km/h) into a maximum workload (in km). It plays the same role as a time step $\Delta t$ does when converting a power (in MW) into an energy (in MWh) in the power system models you will meet later in the course.**

The municipality's night bike lane cleaning problem can be formulated as the following LP: 

$$
  \begin{align}
      \min_{x_i} \quad &\sum_{i=1}^{N} c_i x_i \\
      \textrm{s.t.} \quad &0 \leq x_i \leq v_i H \quad \forall i \in \{1,...,N\} \\
      & \sum_{i=1}^{N} x_i = \sum_{j=1}^{D} L_j \\
  \end{align}
$$

We provide the following input data:

In [11]:
# Define ranges and indexes
N_CONTRACTORS = 3 # number of available contractors
N_DISTRICTS = 2 # number of districts to be cleared
hours = 4 # hours available before the deadline (H)
CONTRACTORS = range(N_CONTRACTORS) # range of contractors
DISTRICTS = range(N_DISTRICTS) # range of districts

# Set values of input parameters
contractor_price = [900,400,1500] # Price of each contractor in DKK/km (c_i)
contractor_speed = [40,30,50] # Clearing speed of each contractor in km/h (v_i)
district_length = [120,80] # Length of bike lane in each district in km (L_j)

- In the same way as in step 2, please initialize and solve the problem using ```gurobipy```.

In [12]:
#create and name a new gurobi model
model = gp.Model(name="Bike Lane Cleaning Problem")

In [13]:
#create decision variables
clearing_variables = [model.addVar(lb=0, ub=float('inf'), vtype=GRB.CONTINUOUS, name=f'clearing by contractor {i}') for i in CONTRACTORS]
#clearing_variables = [model.addVar(lb=0, ub=contractor_speed[i]*hours, vtype=GRB.CONTINUOUS, name=f'clearing by contractor {i}') for i in CONTRACTORS]

In [14]:
#add constraints
coverage_constraint = model.addLConstr(gp.quicksum(clearing_variables[i] for i in CONTRACTORS), GRB.EQUAL, gp.quicksum(district_length[j] for j in DISTRICTS), name='coverage constraint')
capacity_constraints = [model.addLConstr(clearing_variables[i], GRB.LESS_EQUAL, contractor_speed[i]*hours, name=f'capacity constraint {i}') for i in CONTRACTORS]

In [15]:
#add objective function
model.setObjective(gp.quicksum(contractor_price[i]*clearing_variables[i] for i in CONTRACTORS), GRB.MINIMIZE)

In [16]:
#solve optimization problem
model.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.4 LTS")


CPU model: Intel(R) Xeon(R) Processor @ 2.10GHz, instruction set [SSE2|AVX|AVX2|AVX512]


Thread count: 1 physical cores, 1 logical processors, using up to 1 threads


Optimize a model with 4 rows, 3 columns and 6 nonzeros (Min)


Model fingerprint: 0xcec9c235


...

Iteration    Objective       Primal Inf.    Dual Inf.      Time


       0    8.0000000e+04   1.000000e+01   0.000000e+00      0s


       1    1.2000000e+05   0.000000e+00   0.000000e+00      0s


Solved in 1 iterations and 0.01 seconds (0.00 work units)


Optimal objective  1.200000000e+05


In [17]:
LP_printer(model) #check status and print results


-------------------   RESULTS  -------------------
Optimal objective value: 120000.0
Optimal value of clearing by contractor 0: 80.0
Optimal value of clearing by contractor 1: 120.0
Optimal value of clearing by contractor 2: 0.0
Pi value associated with coverage constraint: 900.0
Pi value associated with capacity constraint 0: 0.0
Pi value associated with capacity constraint 1: -500.0
Pi value associated with capacity constraint 2: 0.0


## Section 3: Compact matrix formulation

Linear problems can be expressed in a more general way by defining the inputs before-hand as vectors and mnatrices, and making the rest of the code more general. 

The standard LP matrix formulation is: 

$$
  \begin{align}
      \textrm{minimize} \quad &c^Tx \\
      \textrm{subject to} \quad &Ax \geq b \\
      &x \geq 0 \\
  \end{align}
$$

with: 
- $ N \in \mathbb{N} $ number of decision variables
- $ M \in \mathbb{N} $ number of constraints
- $ c \in \mathbb{R}^N $ cost vector
- $ x \in \mathbb{R}^N $ vector of decision variables
- $ A \in \mathbb{R}^{M \times N}$ matrix of linear coefficients of constraints
- $ b \in \mathbb{R}^{M}$ vector of right-hand side coefficients of constraints

We can reformulate the simple LP defined in Section 2 in this form by defining all the optimization problems' inpout data in a standard format, namely:
- vectors of cost coefficients ```objective_coeff``` $(c)$ and right hand side coefficients ```constraints_rhs``` $(b)$,
- Matrix of input parameters ```constraints_coeff``` $(A)$
- Vector ot constraint directions ```constraints_sense``` ($\leq$ or $\geq$),
- Optimization direction ```optimization_sense``` (minimize or maximize).

In addition, we introduce the following labels for the variables, constraints and the Gurobi model (for clarity): 
- Vectors of variable nsames ```variables_name``` and constraint names```constraints_name```, and model name ```model_name```

In [18]:
# Set values of input parameters and define decision variables names
objective_coeff = [30,20] # Coefficients in objective function
constraints_coeff = [
    [0.6, 0.2],
    [0.4, 0.8]
] # Linear coefficients of constraints 
constraints_rhs = [60, 100] # Right hand side coefficients of constraints
constraints_sense =  [GRB.GREATER_EQUAL, GRB.GREATER_EQUAL] # Direction of constraints
objective_sense = GRB.MINIMIZE # Direction of optimization
variables_name = ['x_1','x_2'] # Names of decision variables
constraints_name = ['constraint_1','constraint_2'] # Names of constraints
model_name = 'My_LP_problem' # Name of model

**Note:  ```constraints_coeff``` is defined as a list of vectors of length equal to the number of constraints (```len(constraints_coeff) = n_constraints```), and such that each element of this list ($a \in$ ```constraints_coeff```) is itself a vector of length equal to the number of variables (```len(a) = n_variables```); i.e. $A = [[A_{11}, ... , A_{1N}] , [A_{M1}, ... , A_{MN}]]$**

- We can also define the number of variables and constraints as:

In [19]:
n_constraints = len(constraints_rhs)
n_variables = len(objective_coeff)
CONSTRAINTS = range(n_constraints)
VARIABLES = range(n_variables)

- We then define a new Gurobi model:

In [20]:
# create model
model = gp.Model(name=model_name)

- We add **positive** decision variables to the Gurobi model:

In [21]:
# Add variables to the Gurobi model
variables = [model.addVar(lb=0, name=variables_name[j]) for j in VARIABLES]

- We define the objective function and optimization direction of the Gurobi model:

In [22]:

# Set objective function and optimization direction of the Gurobi model
objective = gp.quicksum(objective_coeff[j] * variables[j] for j in VARIABLES)         
model.setObjective(objective, objective_sense)

- We add **linear** constraints to the Gurobi model:

In [23]:
constraints = [
                model.addLConstr(
                        gp.quicksum(constraints_coeff[i][j] * variables[j] for j in VARIABLES),
                        constraints_sense[i],
                        constraints_rhs[i],
                        name=constraints_name[i]
                ) for i in CONSTRAINTS
]

- We can now solve this optimization model and compare its results to the ones in Section 2:

In [24]:
# Optimize the Gurobi model

model.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.4 LTS")


CPU model: Intel(R) Xeon(R) Processor @ 2.10GHz, instruction set [SSE2|AVX|AVX2|AVX512]


Thread count: 1 physical cores, 1 logical processors, using up to 1 threads


Optimize a model with 2 rows, 2 columns and 4 nonzeros (Min)


Model fingerprint: 0x20d42a0c


...

Iteration    Objective       Primal Inf.    Dual Inf.      Time


       0    0.0000000e+00   1.600000e+02   0.000000e+00      0s


       2    3.9000000e+03   0.000000e+00   0.000000e+00      0s


Solved in 2 iterations and 0.01 seconds (0.00 work units)


Optimal objective  3.900000000e+03


In [25]:
# Check if the optimization was successful and print solutions
optimal_objective, optimal_variables, optimal_Pis = LP_saver(model)
LP_printer(model)


-------------------   RESULTS  -------------------
Optimal objective value: 3900.0
Optimal value of x_1: 70.0
Optimal value of x_2: 90.0
Pi value associated with constraint_1: 40.0
Pi value associated with constraint_2: 15.0


### Task: Solving the bike-lane cleaning problem

Now, use this general formulation to formulate and solve the bike-lane cleaning problem (see section 3):

**Automated LP Builder**

To automate this process, we can introduce a function that takes all the input paramaters in the standard format defined above, and returns a built model:

In [26]:
def LP_builder(
        objective_coeff: list[float],                # Coefficients in objective function
        constraints_coeff: list[list[float]],    # Linear coefficients of constraints
        constraints_rhs: list[float],                # Right hand side coefficients of constraints
        constraints_sense: list[int],               # Direction of constraints
        objective_sense: int,                           # Direction of optimization
        variables_name: list[str],                # Names of decision variables
        constraints_name: list[str],                # Names of constraints
        model_name: str                                 # Name of model
):
    # Build model
    model = gp.Model(name=model_name)
    

    # add variables
    VARIABLES= range(len(objective_coeff))
    variables = [model.addVar(lb=0, name=variables_name[j]) for j in VARIABLES]

    # Objective
    objective = gp.quicksum(objective_coeff[j] * variables[j] for j in VARIABLES)
    model.setObjective(objective, objective_sense)

    # Constraints
    CONSTRAINTS = range(len(constraints_rhs))
    for i in CONSTRAINTS:
        model.addLConstr(
                gp.quicksum(constraints_coeff[i][j] * variables[j] for j in VARIABLES),
                constraints_sense[i],
                constraints_rhs[i],
                name=constraints_name[i]
        )
    
    model.update()
    
    return model

- We define the input parameters for the bike lane cleaning problem in the standard format:

In [27]:
objective_coeff = contractor_price # Coefficients in objective function
constraints_coeff =  [
    [1,1,1], # full coverage: sum of all x_i
    [1,0,0], # capacity of contractor 0
    [0,1,0], # capacity of contractor 1
    [0,0,1]  # capacity of contractor 2
] # Linear coefficients of constraints
constraints_rhs = [sum(district_length)]+[s*hours for s in contractor_speed] # Right hand side coefficients of constraints
constraints_sense = [GRB.EQUAL] + [GRB.LESS_EQUAL for _ in CONTRACTORS] # direction of constraints
objective_sense = GRB.MINIMIZE # direction of optimization
variables_name = [f'cleaning by contractor {i}' for i in CONTRACTORS] # name of decision variables
constraints_name = ['Full coverage constraint'] + [f'maximum cleaning capacity constraint for contractor {i}' for i in CONTRACTORS] # name of constraints
model_name = 'bike lane cleaning problem' # name of model

- We build the Gurobi model using this function:

In [28]:
model = LP_builder(objective_coeff,constraints_coeff,constraints_rhs,constraints_sense,objective_sense,variables_name,constraints_name,model_name) # Gurobi model

- We solve this Gurobi problem, check its status, save and print its results

In [29]:
# solve model
model.optimize()

# check status and print results
optimal_objective, optimal_variables, optimal_Pis = LP_saver(model)
LP_printer(model)

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.4 LTS")


CPU model: Intel(R) Xeon(R) Processor @ 2.10GHz, instruction set [SSE2|AVX|AVX2|AVX512]


Thread count: 1 physical cores, 1 logical processors, using up to 1 threads


Optimize a model with 4 rows, 3 columns and 6 nonzeros (Min)


Model fingerprint: 0xcec9c235


...

       0    8.0000000e+04   1.000000e+01   0.000000e+00      0s


       1    1.2000000e+05   0.000000e+00   0.000000e+00      0s


Solved in 1 iterations and 0.01 seconds (0.00 work units)


Optimal objective  1.200000000e+05



-------------------   RESULTS  -------------------
Optimal objective value: 120000.0
Optimal value of cleaning by contractor 0: 80.0
Optimal value of cleaning by contractor 1: 120.0
Optimal value of cleaning by contractor 2: 0.0
Pi value associated with Full coverage constraint: 900.0
Pi value associated with maximum cleaning capacity constraint for contractor 0: 0.0
Pi value associated with maximum cleaning capacity constraint for contractor 1: -500.0
Pi value associated with maximum cleaning capacity constraint for contractor 2: 0.0


## Section 4: Introduction to Object-oriented programming (OOP)

### What is OOP?

- OOP is a very powerful tool to structure large optimization problems. 
- In this section, key concepts within OOP are introduced and in the next section, they are applied to the example problem from section 2.

### Classes 
OOP is all about classes. We'll use the class ```Dog``` (below) as a basis to discuss key concepts.

In [30]:
class Dog:

    def __init__(self, breed: str, age: int):
        self.breed = breed
        self.age = age 
    
    def bark(self):
        if self.breed == 'Bloodhound':
            print("WOOF WOOF")
        elif self.breed == "Chihuahua":
            print("woof woof")
        else: 
            raise NotImplementedError("I don't know the bark of this dog")

### Instance
We can create an object which is an instance of the class by providing the arguments ```breed``` and ```age```.

In [31]:
pluto = Dog('Bloodhound', 94)

### ```__init__``` method and attributes
When we created the instance ```pluto```, the ```self.__init__``` method was automatically called <br> 
and the two attributes ```self.breed``` and ```self.age``` were set. Here, ```self```refers to the instance. <br> 
We can access attributes outside of the class with ```instance.attribute```.

In [32]:
print(pluto.breed)
print(pluto.age)

Bloodhound
94


### Methods
Functions defined inside the class are called methods and these can be performed on instances of the class.<br>
The methods often use (or alter) attributes like a dog's bark depends on its breed.

In [33]:
print("Pluto barks: ")
pluto.bark()
harajuku = Dog("Chihuahua", 23)
print("Harajuku barks:")
harajuku.bark()

Pluto barks: 
WOOF WOOF
Harajuku barks:
woof woof


### Inheritance 

One class (let's call it class 1) can "extend" another class (class 2), which means it inherits <br>
the attributes and methods of class 2. Quite fittingly, class 2 is refered to as the parent class <br>
and class 1, the child class. The class definition looks like this: ```class Child(Parent):```. <br>
We continue the dog example below. 

In [34]:
class Chihuahua(Dog):

    def __init__(self, age: int, shake: str):
        self.breed = "Chihuahua"
        self.age = age
        self.shake = shake

In [35]:
tinkerbell = Chihuahua(14, 'strong')
tinkerbell.bark()

woof woof


- Notice how we can use the method ```Dog.bark()``` as it is defined in the parent class ```Dog```,
- and how we introduced a new attribute ```shake``` which is specific to Chihuahuas. 

### Example: Linear optimizatin problem with OOP

We define below a general class of Linear Optimization Problems and methods to initialize, solve and display its results. 

**Admittedly, it is a bit over the top to use OOP for the example problems above. However,<br> 
in the coming exercises/assignments, OOP will be a big help, in particular for solving multiple instances of the same optimization model and running numerical experiements in a structured way.**

- Firstly, we introduce a small class named ```Expando``` which allows for instance attributes to have attributes. (It will make sense later :))

In [36]:
class Expando(object):
    '''
        A small class which can have attributes set
    '''
    pass

- Then, we define an ```InputData``` class which holds the necessary data for the optimization problem. 
- Therefore, it has attributes like ```self.VARIABLES```, ```self.objective_coeff```, ```self.constraints_coeff```, etc.

In [37]:
class LP_InputData:

    def __init__(
        self, 
        objective_coeff: list[float],               # Coefficients in objective function
        constraints_coeff: list[list[float]],    # Linear coefficients of constraints
        constraints_rhs: list[float],                # Right hand side coefficients of constraints
        constraints_sense: list[int],              # Direction of constraints
        objective_sense: int,                           # Direction of optimization
        variables_name: list[str],                      # Names of variables
        constraints_name: list[str],                    # Names of constraints
        model_name: str                                 # Name of model
    ):
        self.objective_coeff = objective_coeff
        self.constraints_coeff = constraints_coeff
        self.constraints_rhs = constraints_rhs
        self.constraints_sense = constraints_sense
        self.objective_sense = objective_sense
        self.variables_name = variables_name
        self.constraints_name = constraints_name
        self.model_name = model_name
        self.VARIABLES = range(len(objective_coeff))
        self.CONSTRAINTS = range(len(constraints_rhs))


- Now, we can define the class ```LP_OptimizationProblem```, which takes an instance of the InputData class as the only argument and stores it as ```self.data```.
- It has methods to build and solve the problem as well as save and display results. 

In [38]:
class LP_OptimizationProblem():

    def __init__(self, input_data: LP_InputData): # initialize class
        self.data = input_data # define data attributes
        self.results = Expando() # define results attributes
        self._build_model() # build gurobi model
    
    def _build_variables(self):
        self.variables = [self.model.addVar(lb=0, name=self.data.variables_name[j]) for j in self.data.VARIABLES]
    
    def _build_constraints(self):
        for i in self.data.CONSTRAINTS:
            self.model.addLConstr(
                gp.quicksum(self.data.constraints_coeff[i][j] * self.variables[j] for j in self.data.VARIABLES),
                self.data.constraints_sense[i],
                self.data.constraints_rhs[i],
                name = self.data.constraints_name[i]
            )

    def _build_objective_function(self):
        objective = gp.quicksum(self.data.objective_coeff[j] * self.variables[j] for j in self.data.VARIABLES)
        self.model.setObjective(objective, self.data.objective_sense)

    def _build_model(self):
        self.model = gp.Model(name=self.data.model_name)
        self._build_variables()
        self._build_objective_function()
        self._build_constraints()
        self.model.update()
    
    def _save_results(self):
        self.results.objective_value = self.model.ObjVal
        self.results.variables = {v.VarName:v.x for v in self.model.getVars()}
        self.results.optimal_Pis = {f' Pi associated with {c.ConstrName}':c.Pi for c in self.model.getConstrs()}

    def run(self):
        self.model.optimize()
        if self.model.status == GRB.OPTIMAL:
            self._save_results()
        else:
            print(f"optimization of {self.model.ModelName} was not successful")
    
    def display_results(self):
        print()
        print("-------------------   RESULTS  -------------------")
        print("Optimal objective:", self.results.objective_value)
        for key, value in self.results.variables.items():
                print(f'Optimal value of {key}:', value)
        for key, value in self.results.optimal_Pis.items():
                print(f'Optimal value of {key}:', value)

- Notice how ```self.results = Expando()``` allows us to save different results in the ```self.results``` attribute, e.g., ```self.results.objective_value```.

#### Task: Solving the Toy model and bike-lane cleaning problems

- We can now create Gurobi models and solve them by creating an instance of the LP_OptimizationProblem class and calling the run() method. The results can be displayed using the display_results() method.
- Below, write what corresponds to the ```main``` function where you create instances of the classes and use their methods to solve their instances. 
- Solve the toy model and the bike-lane clearing model introduced in Section 2 and display their results. 

In [39]:
# This corresponds to the main function

# Define a dictionary containing the 2 instances of the model to be solved: toy problem and bike-lane clearing problem

input_data_instances = {
    ("toy problem","base case"): LP_InputData(
        # Set values of input parameters and define decision variables names
        objective_coeff = [30,20] , # Coefficients in objective function
        constraints_coeff = [
            [0.6, 0.2],
            [0.4, 0.8]
        ] , # Linear coefficients of constraints 
        constraints_rhs = [60, 100] , # Right hand side coefficients of constraints
        constraints_sense =  [GRB.GREATER_EQUAL, GRB.GREATER_EQUAL] , # Direction of constraints
        objective_sense = GRB.MINIMIZE , # Direction of optimization
        variables_name = ['x_1','x_2'] , # Names of decision variables
        constraints_name = ['constraint_1','constraint_2'] , # Names of constraints
        model_name = 'toy problem' # Name of model
    ),
    ("bike lane cleaning problem","base case"): LP_InputData(
        objective_coeff = contractor_price, # Coefficients in objective function
        constraints_coeff =  [
            [1,1,1], # full coverage: sum of all x_i
            [1,0,0], # capacity of contractor 0
            [0,1,0], # capacity of contractor 1
            [0,0,1]  # capacity of contractor 2
        ], # Linear coefficients of constraints
        constraints_rhs = [sum(district_length)]+[s*hours for s in contractor_speed], # Right hand side coefficients of constraints
        constraints_sense = [GRB.EQUAL] + [GRB.LESS_EQUAL for _ in CONTRACTORS], # direction of constraints
        objective_sense = GRB.MINIMIZE, # direction of optimization
        variables_name = [f'cleaning by contractor {i}' for i in CONTRACTORS], # name of decision variables
        constraints_name = ['Full coverage constraint'] + [f'maximum cleaning capacity constraint for contractor {i}' for i in CONTRACTORS], # name of constraints
        model_name = 'bike lane cleaning problem' # name of model
    )
}

models = {key:LP_OptimizationProblem(value) for key, value in input_data_instances.items()}

for model in models.values():
    model.run() # results for all instances of the model created are saved in the model.results attributes
    
for instance, model in models.items():
    print(f'------------------- Results for {instance} -------------------')
    model.display_results()
    print(f'-----------------------------------------------------------------------')

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.4 LTS")


CPU model: Intel(R) Xeon(R) Processor @ 2.10GHz, instruction set [SSE2|AVX|AVX2|AVX512]


Thread count: 1 physical cores, 1 logical processors, using up to 1 threads


Optimize a model with 2 rows, 2 columns and 4 nonzeros (Min)


Model fingerprint: 0x20d42a0c


...

       0    8.0000000e+04   1.000000e+01   0.000000e+00      0s


       1    1.2000000e+05   0.000000e+00   0.000000e+00      0s


Solved in 1 iterations and 0.01 seconds (0.00 work units)


Optimal objective  1.200000000e+05


------------------- Results for ('toy problem', 'base case') -------------------

-------------------   RESULTS  -------------------
Optimal objective: 3900.0
Optimal value of x_1: 70.0
Optimal value of x_2: 90.0
Optimal value of  Pi associated with constraint_1: 40.0
Optimal value of  Pi associated with constraint_2: 15.0
-----------------------------------------------------------------------
------------------- Results for ('bike lane cleaning problem', 'base case') -------------------

-------------------   RESULTS  -------------------
Optimal objective: 120000.0
Optimal value of cleaning by contractor 0: 80.0
Optimal value of cleaning by contractor 1: 120.0
Optimal value of cleaning by contractor 2: 0.0
Optimal value of  Pi associated with Full coverage constraint: 900.0
Optimal value of  Pi associated with maximum cleaning capacity constraint for contractor 0: 0.0
Optimal value of  Pi associated with maximum cleaning capacity constraint for contractor 1: -500.0
Optimal value of  P

### From the matrix formulation to a structured model

The matrix formulation of `LP_InputData` and `LP_OptimizationProblem` is fully general for linear problems: any LP can be written as $\min c^T x$ s.t. $Ax \geq b$. It is nevertheless not the formulation we will use in the rest of the course, for four reasons:

1. **The structure of the problem is lost.** In the bike-lane problem, row 2 of `constraints_coeff` is "the capacity of contractor 1", but nothing in the code says so. In a unit commitment problem with 12 generators and 24 hours, $A$ has thousands of rows: finding "the ramping constraint of generator 7 at hour 13" by its row number is hopeless, and so is checking the code against the mathematical formulation.
2. **The data is pre-digested and position-based.** `constraints_rhs = [200, 160, 120, 200]` mixes lane lengths and capacities, and the capacity $v_i H$ is computed before it enters the model. Adding a contractor or a district means editing $A$, $b$, the senses and the names by hand, and keeping them consistent: one wrong shape in $A$ and the model does not build.
3. **The results are hard to use.** `results.variables` is a dictionary keyed by the name strings; to plot the kilometres cleared per contractor, you have to parse names.
4. **It only covers linear problems.** Later in the course you will add binary variables (unit commitment), quadratic costs and nonlinear constraints (power flow). None of these fits in a coefficient matrix.

The alternative is to make the code mirror the mathematical formulation, element by element:

| Mathematical formulation | Code |
|---|---|
| Sets: contractors $i \in \{1,...,N\}$, districts $j \in \{1,...,D\}$ | lists of labels: `data.CONTRACTORS`, `data.DISTRICTS` |
| Parameters: $c_i$, $v_i$, $L_j$, $H$ | dictionaries keyed by the labels: `data.contractor_price[i]` |
| Variables: $x_i$ | one Gurobi variable per element of the set, stored in a dictionary: `variables.cleared_km[i]` |
| Constraint families: coverage, capacity $\forall i$ | one method per family, constraints stored by index: `constraints.capacity[i]` |
| Objective function | one method |

The steps that are the same for *every* optimization problem (create the Gurobi model, call the builders, solve, check the status, save the results) go into a generic parent class `OptimizationProblem`. The mathematical model itself goes into a child class that only implements the three `_build_*` methods, exactly like `Chihuahua` extended `Dog`. The parent class never looks at what the variables and constraints are, so it works for any problem: linear or not, continuous or integer. This is the structure used for the power system models in the rest of the course.

- The parent class ```OptimizationProblem``` contains the generic steps only. Its three ```_build_*``` methods raise an error on purpose: they must be written in the child class.

In [40]:
class OptimizationProblem:
    """
    Generic optimization problem solved with Gurobi.

    This parent class only contains what is common to *every* optimization problem:
    creating the Gurobi model, calling the builders in the right order, solving,
    checking the status and saving the results.

    Everything that is specific to a given problem (its variables, objective and
    constraints) is written in a child class, by overriding the three _build_* methods.
    """

    def __init__(self, input_data, name: str = "optimization problem"):
        self.data = input_data          # input data of the problem instance
        self.variables = Expando()      # groups of decision variables, e.g. self.variables.cleared_km[i]
        self.constraints = Expando()    # families of constraints, e.g. self.constraints.capacity[i]
        self.results = Expando()        # results, filled in after solving
        self._build_model(name)

    # ---------- problem-specific parts: implemented in the child class ----------

    def _build_variables(self):
        raise NotImplementedError("Define the decision variables in the child class.")

    def _build_objective_function(self):
        raise NotImplementedError("Define the objective function in the child class.")

    def _build_constraints(self):
        raise NotImplementedError("Define the constraints in the child class.")

    # ---------- generic parts: identical for every problem ----------

    def _build_model(self, name: str):
        self.model = gp.Model(name=name)
        self._build_variables()
        self._build_objective_function()
        self._build_constraints()
        self.model.update()

    def run(self):
        self.model.optimize()
        self.results.status = self.model.Status
        if self.model.Status == GRB.OPTIMAL:
            self._save_results()
        else:
            print(f"Optimization of {self.model.ModelName} was not successful (status code {self.model.Status}).")

    def _save_results(self):
        self.results.objective_value = self.model.ObjVal
        # Optimal values of the variables, stored with the same group names and keys as self.variables
        self.results.variables = Expando()
        for name, group in vars(self.variables).items():
            setattr(self.results.variables, name, self._get_values(group, "X"))
        # Dual values, stored with the same family names and keys as self.constraints
        # (only defined for problems with continuous variables and linear constraints)
        self.results.duals = Expando()
        if not self.model.IsMIP and not self.model.IsQCP:
            for name, group in vars(self.constraints).items():
                setattr(self.results.duals, name, self._get_values(group, "Pi"))

    @staticmethod
    def _get_values(group, attribute: str):
        """Read a Gurobi attribute ('X', 'Pi', ...) from a group of Gurobi objects.
        The group can be a dictionary (indexed by a set), a list, or a single object."""
        if isinstance(group, dict):
            return {key: getattr(obj, attribute) for key, obj in group.items()}
        if isinstance(group, list):
            return [getattr(obj, attribute) for obj in group]
        return getattr(group, attribute)

    def display_results(self):
        print("-------------------   RESULTS  -------------------")
        print(f"Optimal objective value: {self.results.objective_value:.2f}")
        print("Optimal values of the variables:")
        for name, values in vars(self.results.variables).items():
            self._print_values(name, values)
        if vars(self.results.duals):
            print("Dual values of the constraints:")
            for name, values in vars(self.results.duals).items():
                self._print_values(name, values)

    @staticmethod
    def _print_values(name: str, values):
        if isinstance(values, dict):
            for key, value in values.items():
                print(f"  {name}[{key}]: {value:.2f}")
        elif isinstance(values, list):
            for index, value in enumerate(values):
                print(f"  {name}[{index}]: {value:.2f}")
        else:
            print(f"  {name}: {values:.2f}")

- ```self.variables```, ```self.constraints```, ```self.results.variables``` and ```self.results.duals``` are ```Expando``` containers whose attributes are the *names* of the variable groups and constraint families. A result is therefore found where its variable or constraint was defined, e.g. ```problem.results.duals.capacity["C2"]```. This is where ```Expando``` pays off.
- ```vars(obj)``` returns the attributes of an object as a dictionary ```{name: value}```, and ```setattr(obj, name, value)``` is ```obj.name = value``` with the name given as a string. They let the parent class loop over groups it knows nothing about.
- Dual values (```Pi```) only exist for problems with continuous variables and linear constraints, hence the ```IsMIP```/```IsQCP``` check. For a MILP, ```results.duals``` simply stays empty.

- The input data class holds the sets and the parameters of one instance, labelled by name and with the units in the comments. Compared to ```LP_InputData```, no coefficient is pre-computed and nothing refers to a position in a matrix.

In [41]:
class BikeLane_InputData:

    def __init__(
        self,
        contractor_price: dict[str, float],   # price of each contractor in DKK/km (c_i)
        contractor_speed: dict[str, float],   # clearing speed of each contractor in km/h (v_i)
        district_length: dict[str, float],    # length of bike lane in each district in km (L_j)
        hours: float,                         # hours available before the deadline (H)
    ):
        # Sets: the labels used as keys of the dictionaries
        self.CONTRACTORS = list(contractor_price.keys())
        self.DISTRICTS = list(district_length.keys())
        # Parameters: dictionaries keyed by the elements of the sets
        self.contractor_price = contractor_price
        self.contractor_speed = contractor_speed
        self.district_length = district_length
        self.hours = hours

- The child class contains the mathematical model and nothing else, with one method per family of constraints:

$$
  \begin{align}
      \min_{x_i} \quad &\sum_{i=1}^{N} c_i x_i \\
      \textrm{s.t.} \quad &\sum_{i=1}^{N} x_i = \sum_{j=1}^{D} L_j \\
      &0 \leq x_i \leq v_i H \quad \forall i \in \{1,...,N\} \\
  \end{align}
$$

In [42]:
class BikeLane_OptimizationProblem(OptimizationProblem):

    def _build_variables(self):
        # x_i >= 0 : kilometres cleared by contractor i
        self.variables.cleared_km = self.model.addVars(
            self.data.CONTRACTORS, lb=0, name="cleared_km"
        )

    def _build_objective_function(self):
        # min sum_i c_i x_i
        total_cost = gp.quicksum(
            self.data.contractor_price[i] * self.variables.cleared_km[i]
            for i in self.data.CONTRACTORS
        )
        self.model.setObjective(total_cost, GRB.MINIMIZE)

    def _build_constraints(self):
        # one method per family of constraints
        self._build_coverage_constraint()
        self._build_capacity_constraints()

    def _build_coverage_constraint(self):
        # sum_i x_i = sum_j L_j
        self.constraints.coverage = self.model.addConstr(
            gp.quicksum(self.variables.cleared_km[i] for i in self.data.CONTRACTORS)
            == sum(self.data.district_length[j] for j in self.data.DISTRICTS),
            name="coverage",
        )

    def _build_capacity_constraints(self):
        # x_i <= v_i H   for all i
        self.constraints.capacity = self.model.addConstrs(
            (
                self.variables.cleared_km[i] <= self.data.contractor_speed[i] * self.data.hours
                for i in self.data.CONTRACTORS
            ),
            name="capacity",
        )

- ```model.addVars(keys, ...)``` creates one variable per element of ```keys``` and returns a ```tupledict```: a dictionary keyed by the set elements (```cleared_km["C2"]```), with Gurobi names ```cleared_km[C2]```. ```model.addConstrs(generator, name=...)``` does the same for a family of constraints. Keys can be any label: strings, integers, or tuples such as ```(generator, hour)``` for two-dimensional families.
- Constraints are written as expressions, ```lhs <= rhs```, ```lhs == rhs``` or ```lhs >= rhs```, with ```model.addConstr```. Unlike ```addLConstr```, ```addConstr``` accepts any expression: linear, quadratic (```x[i] * x[i] <= ...```) and, from Gurobi 12 on, general nonlinear functions (```gurobipy.nlfunc```: exp, log, sin, ...). Integer or binary variables are added with ```vtype=GRB.INTEGER``` or ```vtype=GRB.BINARY``` in ```addVars```. In all these cases, the parent class stays unchanged.
- The objective and the constraints read like the formulation: $\sum_i c_i x_i$ becomes ```gp.quicksum(contractor_price[i] * cleared_km[i] for i in CONTRACTORS)```. For a sum over all elements of a ```tupledict```, ```self.variables.cleared_km.sum()``` is a shortcut.

- The ```main``` solves the same instance as before, now with labelled contractors and districts:

In [43]:
# This corresponds to the main function

# Input data of the instance
input_data = BikeLane_InputData(
    contractor_price={"C1": 900, "C2": 400, "C3": 1500},   # DKK/km (c_i)
    contractor_speed={"C1": 40, "C2": 30, "C3": 50},       # km/h (v_i)
    district_length={"D1": 120, "D2": 80},                 # km (L_j)
    hours=4,                                               # h (H)
)

# Create, solve and display
problem = BikeLane_OptimizationProblem(input_data, name="bike lane cleaning problem")
problem.run()
problem.display_results()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.4 LTS")


CPU model: Intel(R) Xeon(R) Processor @ 2.10GHz, instruction set [SSE2|AVX|AVX2|AVX512]


Thread count: 1 physical cores, 1 logical processors, using up to 1 threads


Optimize a model with 4 rows, 3 columns and 6 nonzeros (Min)


Model fingerprint: 0xcec9c235


...

       0    8.0000000e+04   1.000000e+01   0.000000e+00      0s


       1    1.2000000e+05   0.000000e+00   0.000000e+00      0s


Solved in 1 iterations and 0.01 seconds (0.00 work units)


Optimal objective  1.200000000e+05


-------------------   RESULTS  -------------------
Optimal objective value: 120000.00
Optimal values of the variables:
  cleared_km[C1]: 80.00
  cleared_km[C2]: 120.00
  cleared_km[C3]: 0.00
Dual values of the constraints:
  coverage: 900.00
  capacity[C1]: 0.00
  capacity[C2]: -500.00
  capacity[C3]: 0.00


- The results have the same structure as the model, so a result is found where the corresponding variable or constraint was defined, without parsing any name:

In [44]:
print("km cleared by contractor C2:", problem.results.variables.cleared_km["C2"])
print("dual of the coverage constraint (marginal cost of one more km to clear):", problem.results.duals.coverage)
print("dual of the capacity constraint of C2 (value of one more km of capacity for C2):", problem.results.duals.capacity["C2"])

km cleared by contractor C2:

 120.0
dual of the coverage constraint (marginal cost of one more km to clear): 900.0
dual of the capacity constraint of C2 (value of one more km of capacity for C2): -500.0


What changed, in summary:

| | Matrix formulation | Structured formulation |
|---|---|---|
| Model type | LP only ($Ax \geq b$) | any: LP, MILP, QP, nonlinear |
| Data | anonymous $A$, $b$, $c$ with pre-computed coefficients | named parameters keyed by the set elements |
| Constraints | rows of $A$ | families, one method each, stored by index |
| Results | dictionary keyed by name strings | same structure as the model: `results.variables.cleared_km["C2"]` |
| Adding a contractor | edit $A$, $b$, senses and names consistently | add one entry to each data dictionary |
| Adding a constraint family | edit $A$, $b$, senses and names | add one method |

**Check for yourself:** add a fourth contractor to ```input_data``` and re-run the ```main```: nothing else needs to change. Then count the lines you would have had to change in the matrix formulation of Section 3.

## Section 5: Sensitivity Analysis (optional)

At optimality, observe which inequality constraints are binding (i.e. left hand side = right hand side), and which ones are not binding (i.e. left hand side < or > right hand side). 

Answer the following questions (add code cells when needed):

**Do not re-solve the optimization problem until instructed to in the verification steps below.**

**1. Non-binding constraints**
Pick an inequality constraint that is **not binding** at the optimal solution (e.g., the maximum quantity of cleaning services you can buy from a specific contractor).

a) Hypothesize: if you slightly increase or decrease the right-hand-side (RHS) value of this constraint, how would the optimal decisions and the optimal objective value change? Justify your reasoning.

b) Verify your hypothesis by re-solving the problem with the perturbed RHS value.

**2. Binding constraints**
Now pick an inequality constraint that **is binding** at the optimal solution.

a) Hypothesize: if you slightly increase or decrease the RHS value of this constraint, how would the optimal decisions and the optimal objective value change? Justify your reasoning.

b) Verify your hypothesis by re-solving the problem with the perturbed RHS value.

**3. Interpreting `optimal_Pis`**
For each binding and non-binding inequality constraint in your model, look at the value of `optimal_Pis` associated with it.

a) What do you observe?
b) Based on this, what would you hypothesize about the interpretation of `optimal_Pis`?
c) How could these values be used to do sensitivity analysis without re-solving the optimization problem?